In [1]:
import sys
from operator import itemgetter

from nltk.corpus import wordnet as wn
from nltk.corpus import wordnet_ic

import nltk
import numpy as np

In [2]:
nltk.download('wordnet_ic')

KeyboardInterrupt: 

In [57]:
CLASS_NAME = [
    "person", "bike", "car",
    "motorbike", "airplane", "bus",
    "train", "truck", "boat",
    "traffic_light", "fire_hydrant", "sign",
    "parking_meter", "bench", "bird",
    "cat", "dog", "horse",
    "sheep", "cow", "elephant",
    "bear", "zebra", "giraffe",
    "backpack", "umbrella", "handbag",
    "tie", "suitcase", "frisbee",
    "skis", "snowboard", "ball",
    "kite", "baseball_bat", "baseball_glove",
    "skateboard", "surfboard", "tennis_racket",
    "bottle", "wineglass", "cup",
    "fork", "knife", "spoon",
    "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli",
    "carrot", "hot_dog", "pizza",
    "donut", "cake", "chair",
    "couch", "flowerpot", "bed",
    "dining_table", "toilet", "tv",
    "laptop", "mouse", "remote",       
    "keyboard", "cellular_phone", "microwave",     
    "oven", "toaster", "sink",        
    "refrigerator", "book", "clock",       
    "vase", "scissors", "teddy_bear",   
    "hair_drier", "toothbrush"                              
]

In [58]:
semcor_ic = wordnet_ic.ic('ic-semcor.dat')

In [59]:
knowledeg_matrix = []

for cls1 in CLASS_NAME:
    cls1_w = wn.synsets(cls1)[0]
    mini_knowledge = []
    for cls2 in CLASS_NAME:
        cls2_w = wn.synsets(cls2)[0]
        sim = cls1_w.lin_similarity(cls2_w,
                                 ic=semcor_ic)  # compute lin similarity here
        mini_knowledge.append(sim)
    knowledeg_matrix.append(mini_knowledge)
knowledeg_matrix = np.array(knowledeg_matrix)

In [60]:
np.savetxt("./coco/coco-wordnet.txt", knowledeg_matrix)
np.save("./coco/coco-wordnet.npy", knowledeg_matrix)

In [52]:
cls1_w = wn.synsets("motorbike")[1]
cls2_w = wn.synsets("bicycle")[1]
sim = cls1_w.lin_similarity(cls2_w,
                                 ic=semcor_ic)  # compute lin similarity here
print(sim)

7.710226042098637e-300


In [53]:
BASE_CLASSES = {
    1: [
        'aeroplane', 'bicycle', 'boat', 'bottle', 'car', 'cat', 'chair',
        'diningtable', 'dog', 'horse', 'person', 'pottedplant', 'sheep',
        'train', 'tvmonitor'
    ],
    2: [
        'bicycle', 'bird', 'boat', 'bus', 'car', 'cat', 'chair', 'diningtable',
        'dog', 'motorbike', 'person', 'pottedplant', 'sheep', 'train',
        'tvmonitor'
    ],
    3: [
        'aeroplane', 'bicycle', 'bird', 'bottle', 'bus', 'car', 'chair', 'cow',
        'diningtable', 'dog', 'horse', 'person', 'pottedplant', 'train',
        'tvmonitor'
    ],
}

NOVEL_CLASSES = {
    1: ['bird', 'bus', 'cow', 'motorbike', 'sofa'],
    2: ['aeroplane', 'bottle', 'cow', 'horse', 'sofa'],
    3: ['boat', 'cat', 'motorbike', 'sheep', 'sofa'],
}

In [54]:
split = 1

In [55]:
base_classes = BASE_CLASSES[split]
novel_classes = NOVEL_CLASSES[split]
classes = base_classes + novel_classes

# fix some name alias
cls_map = dict()
for cls in classes:
    if cls == 'diningtable':
        cls_ = 'dining_table'
    elif cls == 'pottedplant':
        cls_ = 'flowerpot'
    elif cls == 'tvmonitor':
        cls_ = 'tv'
    elif cls == 'bicycle':
        cls_ = 'bike'
    else:
        cls_ = cls
    cls_map[cls] = cls_

semcor_ic = wordnet_ic.ic('ic-semcor.dat')
sim_dicts = dict()
for n_cls in novel_classes:
    n_cls_ = cls_map[n_cls]
    n_w = wn.synsets(n_cls_)[0]
    sims = []
    for b_cls in base_classes:
        b_cls_ = cls_map[b_cls]
        b_w = wn.synsets(b_cls_)[0]
        sim = n_w.lin_similarity(b_w,
                                 ic=semcor_ic)  # compute lin similarity here
        sim = round(sim, 3)
        sims.append((b_cls, sim))
    sims = sorted(sims,
                  key=itemgetter(1))[::-1][:5]  # display top5 similar classes
    sim_dicts[n_cls] = sims

for k, v in sim_dicts.items():
    print(k, v)

bird [('horse', 0.753), ('dog', 0.708), ('cat', 0.668), ('sheep', 0.647), ('person', 0.412)]
bus [('train', 0.845), ('car', 0.604), ('aeroplane', 0.575), ('boat', 0.561), ('bottle', 0.364)]
cow [('horse', 0.795), ('sheep', 0.792), ('dog', 0.68), ('cat', 0.645), ('person', 0.342)]
motorbike [('bicycle', 1.0), ('tvmonitor', 0.0), ('train', 0.0), ('sheep', 0.0), ('pottedplant', 0.0)]
sofa [('chair', 0.867), ('diningtable', 0.596), ('car', 0.431), ('bottle', 0.409), ('aeroplane', 0.407)]
